## 第 6 课：同一数据两次归约

题目：[Triton: Compute Per-Row Mean and Variance](https://www.deep-ml.com/problems/972?from=Triton%20Essentials)（ID 972）

计算目标：

In [ ]:
mean[m] = (1 / N) * sum_n x[m, n]
var[m]  = (1 / N) * sum_n (x[m, n] - mean[m]) ** 2

返回两个一维 Tensor `(mean, var)`，长度都是 `M`。方差是**总体方差**（population variance，除以 N）。

例如：

In [ ]:
x = [[1, 2, 3],
     [4, 6, 8]]

mean = [2.0, 6.0]
var  = [0.6667, 2.6667]   # 总体方差

和第 5 课一样：一个 program 一行，整行装进寄存器。区别是同一份加载的数据要做**两次归约**。

### 1. 先算 mean

In [ ]:
row = tl.load(..., mask=mask, other=0.0)
mean = tl.sum(row, axis=0) / N

注意两点：

- 除以的 `N` 是**运行时参数**（真实行宽），不是 `BLOCK_SIZE_N`（可能更大）。
- padding 的 lane 贡献 0，所以 `tl.sum(row)` 就是有效元素的和。

### 2. 再算 var：必须重新 mask

In [ ]:
diff = row - mean
sq = diff * diff
sq = tl.where(mask, sq, 0.0)   # 关键！把 padding lane 的平方清零
var = tl.sum(sq, axis=0) / N

为什么必须重新 mask？因为 padding 的 lane 加载出来是 `0.0`，但 `(0.0 - mean)² = mean² ≠ 0`，会污染方差的和。

而求 mean 时不需要重新 mask：padding 的 `0.0` 贡献就是 0，没有影响。

### 3. 两个输出 buffer

每个 program 写两个标量到两个不同的 buffer：

In [ ]:
tl.store(mean_ptr + pid, mean)
tl.store(var_ptr + pid, var)

## 你的代码骨架

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def mean_var_kernel(
    x_ptr,
    mean_ptr,
    var_ptr,
    M,
    N,
    stride_xm,
    BLOCK_SIZE_N: tl.constexpr,
):
    # TODO 1：行 ID / offs_n / mask / other=0.0 加载整行（同 row_sum）

    # TODO 2：mean = tl.sum(row, axis=0) / N
    # 注意：除以 N（真实长度），不是除以 BLOCK_SIZE_N

    # TODO 3：计算 (row - mean) ** 2
    # 然后重新 mask：tl.where(mask, sq, 0.0)
    # 为什么？padding lane 的值是 0，但 (0 - mean)^2 != 0，会污染方差

    # TODO 4：var = tl.sum(sq, axis=0) / N

    # TODO 5：把 mean、var 分别写入两个输出 buffer（下标都是 pid）
    pass


def mean_var(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    # TODO 6：取得 M、N

    # TODO 7：BLOCK_SIZE_N = triton.next_power_of_2(N)

    # TODO 8：分配 mean、var（形状都是 (M,)）

    # TODO 9：创建一维 grid（M,）

    # TODO 10：启动 kernel

    # TODO 11：返回 (mean, var)
    pass

同时回答：

1. 为什么求方差时要把 `(row - mean) ** 2` 重新 mask（`tl.where(mask, sq, 0.0)`），而求均值时不需要？
2. 为什么 mean 和 var 都除以 `N` 而不是除以 `BLOCK_SIZE_N`？（例如 N=100, BLOCK_SIZE_N=128 时除以哪个才对？）
3. 题目要求的是总体方差还是样本方差？两者差在哪？

把代码和三个答案发给我，我继续审查。